In [2]:
import os
import time
import threading
from pymycobot.mycobot280 import MyCobot280
from pymycobot.genre import Angle, Coord

## 로봇 연결

JetCobot
이미지에는 로봇의 Udev 설정이 돼있다.

In [3]:
mc = MyCobot280('/dev/ttyJETCOBOT', 1000000)
mc.thread_lock = True
print("로봇이 연결되었습니다.")

로봇이 연결되었습니다.


## 로봇의 현재 데이터 읽기

가장 많이 쓸 코드 중 하나

로봇 오른팔

현재 각도: [2.63, -137.81, -15.9, -23.29, 96.94, 46.93]

현재 좌표: [126.9, -10.5, -109.4, 89.49, -43.96, 176.05]

인코더: [2018, 3616, 1867, 2313, 945, 1514]

라디안: [0.046, -2.405, -0.278, -0.406, 1.692, 0.819]

In [ ]:
#현재 각도 읽기
angles = mc.get_angles()
print("현재 각도:",angles)
# 현재 좌표 읽기
coords = mc.get_coords()
print("현재 좌표:", coords)
# 인코더 값 읽기
encoders = mc.get_encoders()
print("인코더:", encoders)
# 라디안 값 읽기
radians = mc.get_radians()
print("라디안:", radians)

## 각 조인트의 동작 범위 확인하기

In [4]:
ANGLE_MIN = [-168, -135, -150, -145, -165, -180, 0]
ANGLE_MAX = [168, 135, 150, 145, 165, 180, 100]
for i in range(7):
    print(f"관절 {i+1}: {ANGLE_MIN[i]} ~ {ANGLE_MAX[i]}도")

관절 1: -168 ~ 168도
관절 2: -135 ~ 135도
관절 3: -150 ~ 150도
관절 4: -145 ~ 145도
관절 5: -165 ~ 165도
관절 6: -180 ~ 180도
관절 7: 0 ~ 100도


## 로봇을 초기위치로 이동

로봇팔에는 보통 Home 포즈라고 부르는 동작이 있다

In [9]:
# 로봇을 초기 위치로 리셋
initial_angles = [0, 0, 0, 0, 0, 0]
#initial_angles = [-77.78, 18.89, -103.62, 70.48, 5.53, 7.73]
speed = 50

print("로봇을 초기 위치로 리셋합니다.")
mc.send_angles(initial_angles, speed)
mc.set_gripper_value(100, speed) # 그리퍼 열기
time.sleep(3) # 움직임이 완료될 때까지 대기

print("리셋 완료")

로봇을 초기 위치로 리셋합니다.
리셋 완료


## 단일 관절 각도 움직이기

send_angle(조인트 번호, 목표 각도, 속도)

In [22]:
# 관절 1(베이스)을 30도로 이동
joint_id = Angle.J1.value # 관절 1 (베이스)
angle = 0
speed = 50

print(f"관절 {joint_id}를 {angle}도로 이동합니다.")
mc.send_angle(joint_id, angle, speed)
time.sleep(0.5) # 움직임이 완료될 때까지 대기

# 관절 1을 다시 0도로 복귀
'''print(f"관절 {joint_id}를 0도로 복귀합니다.")
mc.send_angle(joint_id, 0, speed)
time.sleep(2) # 움직임이 완료될 때까지 대기'''

관절 1를 0도로 이동합니다.


'print(f"관절 {joint_id}를 0도로 복귀합니다.")\nmc.send_angle(joint_id, 0, speed)\ntime.sleep(2) # 움직임이 완료될 때까지 대기'

## 모든 관절 각도 움직이기

send_angles(목표 각도 List, 속도)

In [25]:
# 모든 관절을 지정된 각도로 이동
target_angles = [20, 20, -20, 20, 20, -45]
speed = 50

print(f"모든 관절을 {target_angles}로 이동합니다.")
mc.send_angles(target_angles, speed)
time.sleep(3) # 움직임이 완료될 때까지 대기

# 초기 위치로 복귀
print("초기 위치로 복귀합니다.")
mc.send_angles([0, 0, 0, 0, 0, 0], speed)
time.sleep(3) # 움직임이 완료될 때까지 대기

모든 관절을 [20, 20, -20, 20, 20, -45]로 이동합니다.
초기 위치로 복귀합니다.


## 좌표로 로봇 제어하기

send_coords(목표 좌표 List, 속도, 모드)

In [19]:
# 현재 좌표 확인
current_coords = mc.get_coords()
print("현재 좌표:", current_coords)

# 1. 먼저 Z축을 낮추기
work_coords = current_coords.copy()
work_coords[2] -= 50 # Z를 50mm 내리기
print(f"Z축을 {work_coords[2]}로 내립니다.")
mc.send_coords(work_coords, 10, 0)

time.sleep(7)
current_coords = mc.get_coords()
print("현재 좌표:", current_coords)


# 2. X 좌표 이동
x_coords = work_coords.copy()
x_coords[0] += 50 # X + 20mm
print(f"X 좌표를 {x_coords[0]}로 이동합니다.")
mc.send_coords(x_coords, 10, 0)
time.sleep(5)

current_coords = mc.get_coords()
print("현재 좌표:", current_coords)

# 3. Y 좌표 이동
y_coords = x_coords.copy()
y_coords[1] -= 50 # Y - 20mm
print(f"Y 좌표를 {y_coords[1]}로 이동합니다.")
mc.send_coords(y_coords, 10, 0)
time.sleep(5)

current_coords = mc.get_coords()
print("현재 좌표:", current_coords)

# 4. 최종 좌표 확인
final_coords = mc.get_coords()
print("최종 좌표:", final_coords)

# 5. 초기 위치로 복귀
print("초기 위치로 복귀합니다.")
mc.send_angles([0, 0, 0, 0, 0, 0], 50)
time.sleep(3)
print("초기 위치 복귀 완료")

현재 좌표: [48.0, -64.3, 411.1, -89.56, -0.26, -89.65]
Z축을 361.1로 내립니다.
현재 좌표: [49.3, -64.2, 360.9, -90.17, -0.26, -89.55]
X 좌표를 98.0로 이동합니다.
현재 좌표: [99.9, -64.4, 359.7, -90.79, -0.25, -89.55]
Y 좌표를 -114.3로 이동합니다.
현재 좌표: [102.1, -116.0, 359.0, -90.78, 0.38, -89.47]
최종 좌표: [102.1, -116.0, 359.0, -90.78, 0.38, -89.47]
초기 위치로 복귀합니다.
초기 위치 복귀 완료


## 모든 좌표로 한번에 이동

IK를 풀 수 있을 경우에만 이동한다..!

In [9]:
# 현재 좌표 확인
current_coords = mc.get_coords()
print("현재 좌표:", current_coords)

# 목표 좌표 설정 (현재에서 조금 변경)
target_coords = current_coords.copy()
target_coords[0] += 30 # X + 30mm
target_coords[1] -= 30 # Y - 30mm
target_coords[2] -= 50 # Z - 50mm

print(f"목표 좌표로 이동합니다: {target_coords}")
mc.send_coords(target_coords, 50, 0)
time.sleep(3)

# 초기 좌표로 복귀
print("초기 위치로 복귀합니다.")
mc.send_angles([0, 0, 0, 0, 0, 0], 50)

현재 좌표: [48.6, -64.3, 410.7, -90.08, -0.08, -89.64]
목표 좌표로 이동합니다: [78.6, -94.3, 360.7, -90.08, -0.08, -89.64]
초기 위치로 복귀합니다.


-1

## 그리퍼 제어

무언가를 집어야한다면 그냥 완전히 닫으면 닫기

In [10]:
# 그리퍼 완전히 열기
print("그리퍼를 완전히 엽니다.")
mc.set_gripper_value(100, 50)
time.sleep(1)

# 그리퍼 반쯤 닫기
print("그리퍼를 반쯤 닫습니다.")
mc.set_gripper_value(50, 50)
time.sleep(1)

# 그리퍼 더 닫기
print("그리퍼를 더 닫습니다.")
mc.set_gripper_value(30, 50)
time.sleep(1)

# 그리퍼 완전히 닫기
print("그리퍼를 완전히 닫습니다.")
mc.set_gripper_value(0, 50)
time.sleep(1)

# 그리퍼 다시 열기
print("그리퍼를 다시 엽니다.")
mc.set_gripper_value(100, 50)
time.sleep(1)

그리퍼를 완전히 엽니다.
그리퍼를 반쯤 닫습니다.
그리퍼를 더 닫습니다.
그리퍼를 완전히 닫습니다.
그리퍼를 다시 엽니다.


## 수동 조작 모드 (주의 : 실행전에 손으로 로봇을 잡고 시작해주세요.)

In [10]:
# 모터 비활성화
time.sleep(4)
print("전체 모터를 비활성화합니다.")
mc.release_all_servos()
time.sleep(10)

# 모터 활성화
print("전체 모터를 활성화합니다.")
mc.focus_all_servos()
time.sleep(1)

#현재 각도 읽기
angles = mc.get_angles()
print("현재 각도:",angles)
# 현재 좌표 읽기
coords = mc.get_coords()
print("현재 좌표:", coords)
# 인코더 값 읽기
encoders = mc.get_encoders()
print("인코더:", encoders)
# 라디안 값 읽기
radians = mc.get_radians()
print("라디안:", radians)

전체 모터를 비활성화합니다.
전체 모터를 활성화합니다.
현재 각도: [6.32, -3.86, -44.29, -23.2, 2.72, 43.41]
현재 좌표: [169.6, -43.8, 282.7, -168.0, 14.61, -125.09]
인코더: [1976, 2092, 1544, 2312, 2017, 1554]
라디안: [0.11, -0.067, -0.773, -0.405, 0.047, 0.758]


In [8]:
print("전체 모터를 비활성화합니다.")
mc.release_all_servos()
time.sleep(5)

print("전체 모터를 활성화합니다.")
mc.focus_all_servos()
time.sleep(1)

전체 모터를 비활성화합니다.


전체 모터를 활성화합니다.
